# Calibration design — LHS, score, keep the best

This page opens the Phase J calibration workflow. It builds a Latin-hypercube
design over the priors, scores every point with a memory-bounded batch map, and
keeps the best sixteen. The claim to check:

1. The best 16 design points **bracket the true infection rate** that generated
   the synthetic observations (min ≤ truth ≤ max).

A scatter of log density against infection rate shows the scored design and
highlights those sixteen points.


In [ ]:
from typing import NamedTuple

import numpy as np
import pandas as pd
import plotly.io as pio

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

from summer4 import (
    Compartments,
    FlowModel,
    Property,
    PropertyData,
    PropertyMap,
    SavePlan,
    SaveRequest,
    Target,
    TargetSet,
    TransitionFlow,
    derived_refs,
)
from summer4.epi.calibration import (
    BayesianModel,
    NormalLikelihood,
    Uniform,
    workflow as wf,
)


## Synthetic SIR truth and a BayesianModel

Compile an SIR, observe sparse infecteds at a known infection rate, and wrap
the compiled model in a `BayesianModel` with a Uniform prior on infection.


In [ ]:
class Rates(NamedTuple):
    infection: float
    recovery: float


TRUE_INFECTION = 0.35
TRUE_RECOVERY = 0.1
times = np.array([0.0, 20.0, 40.0, 60.0])

state = Property("state", ("S", "I", "R"))
pmap = PropertyMap.from_property(state)
refs = derived_refs(Rates)
model = FlowModel(pmap)
model.add_flow(TransitionFlow("infection", state["S"], state["I"], refs.infection))
model.add_flow(TransitionFlow("recovery", state["I"], state["R"], refs.recovery))
cm = model.compile()
y0 = PropertyData.wrap(pmap, np.array([999.0, 1.0, 0.0]))
qty = Compartments(where=state["I"])
truth_params = {"infection": TRUE_INFECTION, "recovery": TRUE_RECOVERY}
truth = cm.run(
    truth_params,
    y0,
    t0=0.0,
    t1=80.0,
    dt=1.0,
    save=SavePlan(requests={"I": SaveRequest(qty, ts=times)}),
    solver="euler",
)
raw = truth["I"].at_times(times).values
obs = np.asarray(raw.data if hasattr(raw, "data") else raw).reshape(-1)

targets = TargetSet(
    targets=(
        Target(
            key="I",
            times=times,
            values=obs,
            quantity=qty,
            likelihood=NormalLikelihood(sd=5.0),
        ),
    )
)
bm = BayesianModel(
    cm,
    {"recovery": TRUE_RECOVERY},
    priors=(Uniform("infection", 0.05, 1.0),),
    targets=targets,
    y0=y0,
    run_kwargs={"t0": 0.0, "t1": 80.0, "dt": 1.0, "solver": "euler"},
)


## Latin hypercube, score, keep the best 16

Draw 256 LHS points, evaluate them in batches of 64, and keep the top 16 by
joint log density. Failed solves (if any) are excluded from `best`.


In [ ]:
design = wf.evaluate(bm, wf.lhs(bm, 256, seed=0), batch_size=64)
best16 = design.best(16)
frame = design.to_frame()
best_frame = best16.to_frame()

assert len(design) == 256
assert len(best16) == 16
assert bool(np.all(np.asarray(best16.ok)))


## Log density against infection rate

The full design is a cloud; the best 16 are marked. The vertical line is the
true infection rate. Those sixteen should sit on either side of it.


In [ ]:
plot_df = pd.DataFrame(
    {
        "infection": frame["infection"],
        "log_density": frame["log_density"],
        "which": np.where(
            frame["infection"].isin(best_frame["infection"]),
            "best 16",
            "design",
        ),
    }
)
fig = plot_df.plot.scatter(
    x="infection",
    y="log_density",
    color="which",
    title="LHS design: log density vs infection rate",
)
fig.update_layout(xaxis_title="infection rate", yaxis_title="log density")
fig.add_vline(x=TRUE_INFECTION, line_dash="dash", line_color="black")
fig.show()

inf_best = np.asarray(best16.params["infection"])
assert inf_best.min() <= TRUE_INFECTION <= inf_best.max(), (
    f"best 16 infection rates [{inf_best.min():.3f}, {inf_best.max():.3f}] "
    f"do not bracket truth {TRUE_INFECTION}"
)
print(
    f"best 16 infection in [{inf_best.min():.3f}, {inf_best.max():.3f}]; "
    f"truth={TRUE_INFECTION}"
)
